In [ ]:
import pandas as pd
import os

df_global = pd.read_parquet("/data/big/fmoss/data/model_output/merged/adm1_grouped/2_stage_model_loocv.parquet")
df_global["GID_0"] = df_global["DisNo."].apply(lambda x: x[-3:])

df_geo = pd.read_parquet("/data/big/fmoss/data/model_output/merged/adm1_grouped/model_out_loocv_2_stg_model_geo_adm1.parquet")
df_geo["GID_0"] = df_geo["DisNo."].apply(lambda x: x[-3:])

In [2]:
aux = df_global[["GID_0"]].drop_duplicates().reset_index(drop=True)

import country_converter as coco
# Convert to continent
aux['continent'] = coco.convert(names=aux['GID_0'], to='continent')

# Convert to UN region
aux['region'] = coco.convert(names=aux['GID_0'], to='UNregion')

un_region_to_basin = {
    "Caribbean": "North Atlantic",             # hurricanes in Atlantic/Caribbean
    "Central America": "North Atlantic",       # hurricanes, some Eastern Pacific
    "South America": "North Atlantic",         # north part affected by Atlantic
    "Northern America": "North Atlantic",      # US, Mexico Atlantic side
    "Australia and New Zealand": "Australian Region",  # AUS basin storms
    "Southern Asia": "North Indian",           # Bay of Bengal / Arabian Sea
    "Eastern Asia": "Western Pacific",         # typhoons
    "South-eastern Asia": "Western Pacific",   # typhoons
    "Western Asia": "North Indian",            # Arabian Sea
    "Eastern Africa": "South-West Indian",     # storms in SW Indian
    "Southern Africa": "South-West Indian",    # storms in SW Indian
    "Melanesia": "South Pacific",              # Fiji region storms
    "Micronesia": "Western Pacific",           # typhoons
    "Polynesia": "South Pacific",              # cyclones in SP
    "Southern Europe": "Europe",               # essentially no tropical cyclones in europe, so we define it
}
aux["cyclone_basin"] = aux["region"].map(un_region_to_basin)

df_global = df_global.merge(aux, on="GID_0")
df_geo = df_geo.merge(aux, on="GID_0")

In [6]:
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score
import numpy as np

def compute_basin_metrics(
    df,
    pred_col="predicted",
    true_col="reported",
    basin_col="cyclone_basin",
    thresholds=(0, 15)
):
    """
    Compute F1, precision, recall (binary) and MAE (continuous)
    grouped by cyclone basin.

    Binary metrics are computed for each threshold.
    """

    results = []

    for basin, g in df.groupby(basin_col):

        row = {"cyclone_basin": basin}

        # Absolute error (continuous)
        abs_err = (g[pred_col] - g[true_col]).abs()
        row["mae"] = abs_err.mean()

        for thr in thresholds:
            y_true = (g[true_col] > thr).astype(int)
            y_pred = (g[pred_col] > thr).astype(int)

            suffix = f"thr_{thr}"

            row[f"f1_{suffix}"] = f1_score(y_true, y_pred, zero_division=np.nan)
            row[f"precision_{suffix}"] = precision_score(y_true, y_pred, zero_division=np.nan)
            row[f"recall_{suffix}"] = recall_score(y_true, y_pred, zero_division=np.nan)

        results.append(row)

    return pd.DataFrame(results).sort_values("cyclone_basin")

basin_metrics_global = compute_basin_metrics(df_global)
basin_metrics_geo = compute_basin_metrics(df_geo)

In [7]:
basin_metrics_global

,cyclone_basin,mae,f1_thr_0,precision_thr_0,recall_thr_0,f1_thr_15,precision_thr_15,recall_thr_15
0,Australian Region,0.859609,0.619048,0.541667,0.722222,0.000000,NaN,0.000000
1,North Atlantic,4.747022,0.474377,0.337047,0.800567,0.305263,0.205674,0.591837
2,North Indian,2.579147,0.439141,0.319444,0.702290,0.281250,0.214286,0.409091
3,South Pacific,9.057643,0.524590,0.387097,0.813559,0.376812,0.254902,0.722222
4,South-West Indian,3.499060,0.592405,0.460630,0.829787,0.129032,0.090909,0.222222
5,Western Pacific,3.403294,0.384790,0.277328,0.628219,0.188406,0.118847,0.454301


In [8]:
basin_metrics_geo

,cyclone_basin,mae,f1_thr_0,precision_thr_0,recall_thr_0,f1_thr_15,precision_thr_15,recall_thr_15
0,Australian Region,0.993651,0.326923,0.197674,0.944444,0.000000,NaN,0.000000
1,North Atlantic,4.801625,0.475682,0.338535,0.799622,0.379888,0.261538,0.693878
2,North Indian,1.493276,0.450128,0.338462,0.671756,0.368421,0.437500,0.318182
3,South Pacific,5.822041,0.562500,0.521739,0.610169,0.363636,0.400000,0.333333
4,South-West Indian,1.725087,0.596774,0.691589,0.524823,0.000000,NaN,0.000000
5,Western Pacific,3.134779,0.378317,0.267445,0.646208,0.219634,0.288210,0.177419
